# Standard PDF extraction → NGAT DataFrames

Place `standard_clauses.json` (created from the companion prompt) next to this notebook and run all cells. The notebook preserves a **one-row-per-clause** DataFrame and generates a second DataFrame matching NGAT's `standard_texts_r` columns. No database connection or modification is made.

NGAT currently stores `section` and `subsection` as integers. For example, clauses `8.4.1` and `8.4.2` are grouped under `(section=8, subsection=4)`, with their complete numbers in the formatted text. Section-only clauses go into subsection `0` so that their text is never silently discarded; review these rows before import.

In [ ]:
from pathlib import Path
import json
import re
import pandas as pd

INPUT_FILE = Path('standard_clauses.json')  # Change this if the file is elsewhere.
STANDARD_ID = None  # Set to the existing NGAT standards_r ID before importing the CSV.
OUTPUT_DIR = INPUT_FILE.resolve().parent
CLAUSE_NUMBER = re.compile(r'\d+(?:\.\d+)*')

if STANDARD_ID is not None and (type(STANDARD_ID) is not int or STANDARD_ID <= 0):
    raise ValueError('STANDARD_ID must be an existing positive integer ID, or None for review.')


In [ ]:
# Parse and validate the extraction without changing any clause wording.
payload = json.loads(INPUT_FILE.read_text(encoding='utf-8-sig'))
if not isinstance(payload, dict) or not isinstance(payload.get('clauses'), list):
    raise ValueError('Expected a JSON object with a clauses array.')
if not payload['clauses']:
    raise ValueError('The clauses array is empty. Check the PDF extraction.')

records = []
for index, clause in enumerate(payload['clauses'], start=1):
    if not isinstance(clause, dict):
        raise ValueError(f'Clause {index} is not an object.')
    number, title, body = (str(clause.get(key) or '').strip()
                           for key in ('number', 'title', 'text'))
    if not number or not (title or body):
        raise ValueError(f'Clause {index} is missing its number or both title and text.')
    records.append({'number': number, 'title': title, 'text': body,
                    'page': clause.get('page')})

clauses_df = pd.DataFrame.from_records(records)
duplicates = clauses_df.loc[clauses_df['number'].duplicated(keep=False), 'number'].tolist()
if duplicates:
    raise ValueError(f'Duplicate clause numbers (combine PDF page continuations): {duplicates}')

def sort_key(number):
    if CLAUSE_NUMBER.fullmatch(number):
        return (0, tuple(int(part) for part in number.split('.')))
    return (1, number)  # Keep nonnumeric identifiers visible for manual review.

clauses_df = (clauses_df.assign(_sort=clauses_df['number'].map(sort_key))
              .sort_values('_sort', kind='stable').drop(columns='_sort').reset_index(drop=True))
standard = str(payload.get('standard') or 'Unknown standard')
issues = payload.get('issues', [])
print(f'Standard: {standard} | Extracted clauses: {len(clauses_df)}')
if issues:
    print('PDF extraction issues — REVIEW before importing:')
    print(json.dumps(issues, ensure_ascii=False, indent=2))
blank_text = clauses_df.loc[clauses_df['text'].eq(''), 'number'].tolist()
if blank_text:
    print(f'Clauses with heading only (check these): {blank_text}')
display(clauses_df.head(20))


In [ ]:
# Map the clause hierarchy to NGAT's integer (section, subsection) pairs.
numeric_mask = clauses_df['number'].map(lambda n: CLAUSE_NUMBER.fullmatch(n) is not None)
unmapped_df = clauses_df.loc[~numeric_mask].copy()
if not unmapped_df.empty:
    print('WARNING: Nonnumeric clauses cannot map to NGAT integer columns:')
    print(unmapped_df['number'].tolist())

groups = {}
for clause in clauses_df.loc[numeric_mask].itertuples(index=False):
    parts = [int(part) for part in clause.number.split('.')]
    section, subsection = parts[0], (parts[1] if len(parts) > 1 else 0)
    heading = '#' * min(len(parts) + 1, 6) + f' {clause.number}'
    if clause.title:
        heading += f' {clause.title}'
    formatted = heading + (f'\n\n{clause.text}' if clause.text else '')
    groups.setdefault((section, subsection), []).append(formatted)

ngat_df = pd.DataFrame([
    {'standardid': STANDARD_ID, 'section': section, 'subsection': subsection,
     'text': '\n\n'.join(group)}
    for (section, subsection), group in sorted(groups.items())
], columns=['standardid', 'section', 'subsection', 'text'])
ngat_df = ngat_df.astype({'standardid': 'Int64', 'section': 'int64', 'subsection': 'int64'})
print(f'NGAT grouped rows: {len(ngat_df)}')
if STANDARD_ID is None:
    print('REVIEW ONLY: standardid is blank. Set STANDARD_ID to the correct existing ID before import.')
if (ngat_df['subsection'] == 0).any():
    print('REVIEW: subsection 0 contains section-level text; check its display in Conduct Audit.')
display(ngat_df.head(20))


In [ ]:
# Save UTF-8 CSVs for inspection; this does NOT execute SQL or import anything.
clauses_file = OUTPUT_DIR / 'standard_clauses_review.csv'
ngat_file = OUTPUT_DIR / 'ngat_standard_texts_review.csv'
clauses_df.to_csv(clauses_file, index=False, encoding='utf-8-sig')
ngat_df.to_csv(ngat_file, index=False, encoding='utf-8-sig')
print(f'Saved {clauses_file.name} and {ngat_file.name}')
if not unmapped_df.empty:
    unmapped_file = OUTPUT_DIR / 'unmapped_clauses_review.csv'
    unmapped_df.to_csv(unmapped_file, index=False, encoding='utf-8-sig')
    print(f'Saved {unmapped_file.name} (requires manual handling).')
